In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np


YEARS = range(2009, 2026)
BASE = "https://sgschooling.com/year/{year}/all"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def clean_text(text: str) -> str:
    return " ".join(text.split())

def parse_regular_cell(text: str) -> int:
    """
    For Vacancy and Applied rows.
    Examples:
    '53' -> 53
    '-'  -> 0
    """
    s = clean_text(text)
    if not s or s == "-":
        return 0

    m = re.search(r"\b\d+\b", s)
    return int(m.group()) if m else 0

def parse_taken_cell(text: str) -> int:
    """
    For Taken rows.
    Examples:
    '65 SC1-2 17/12' -> 65
    '61 SC<1' -> 61
    '-' -> 0
    """
    s = clean_text(text)
    if not s or s == "-":
        return 0

    # first standalone integer is the phase-level taken count
    m = re.search(r"(?<!/)\b\d+\b(?!/)", s)
    if m:
        return int(m.group())

    # fallback if only ratio appears
    r = re.search(r"(\d+)\s*/\s*(\d+)", s)
    if r:
        return int(r.group(2))

    return 0

def detect_phase_layout(header_cells):
    """
    Two possible layouts:
    1. Phase 1, 2A, 2B, 2C, 2C(S), 3
    2. Phase 1, 2A(1), 2A(2), 2B, 2C, 2C(S), 3
    """
    cells = [clean_text(x) for x in header_cells]

    if "2A(1)" in cells and "2A(2)" in cells:
        return ["Phase 1", "2A(1)", "2A(2)", "2B", "2C", "2C(S)", "3"]
    return ["Phase 1", "2A", "2B", "2C", "2C(S)", "3"]

def extract_header_phases(soup):
    for tr in soup.select("tr"):
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue

        texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
        if texts and texts[0] == "School":
            return detect_phase_layout(texts[1:])

    raise ValueError("Could not find header row.")

def is_school_row(first: str) -> bool:
    s = clean_text(first)

    if not s:
        return False
    if s.startswith("↳"):
        return False
    if s in {"School", "P1 Ballot History", "All Primary Schools"}:
        return False
    if "Vacancy" in s or "Applied" in s or "Taken" in s:
        return False

    return True

def extract_phase_values(texts, phases, row_type):
    """
    Map row values into a phase:value dictionary.
    """
    vals = {}
    raw_cells = texts[1:1 + len(phases)]

    for phase, cell in zip(phases, raw_cells):
        if row_type == "taken":
            vals[phase] = parse_taken_cell(cell)
        else:
            vals[phase] = parse_regular_cell(cell)

    for p in phases:
        vals.setdefault(p, 0)

    return vals

def parse_year(year: int):
    url = BASE.format(year=year)
    html = requests.get(url, headers=HEADERS, timeout=40).text
    soup = BeautifulSoup(html, "html.parser")

    phases = extract_header_phases(soup)

    out = []
    current_school = None
    vacancy_vals = None
    applied_vals = None
    taken_vals = None

    for tr in soup.select("tr"):
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue

        texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
        if not texts:
            continue

        first = texts[0]

        # school row
        if is_school_row(first):
            current_school = first
            vacancy_vals = None
            applied_vals = None
            taken_vals = None
            continue

        if not current_school:
            continue

        # vacancy row
        if "Vacancy" in first:
            vacancy_vals = extract_phase_values(texts, phases, row_type="vacancy")
            continue

        # applied row
        if "Applied" in first:
            applied_vals = extract_phase_values(texts, phases, row_type="applied")
            continue

        # taken row
        if "Taken" in first:
            taken_vals = extract_phase_values(texts, phases, row_type="taken")

            if vacancy_vals is None:
                vacancy_vals = {p: 0 for p in phases}
            if applied_vals is None:
                applied_vals = {p: 0 for p in phases}
            if taken_vals is None:
                taken_vals = {p: 0 for p in phases}

            # overall year ballot: 2B2C phase with applied > taken
            app_2B = applied_vals.get("2B", 0)
            app_2C = applied_vals.get("2C", 0)

            vac_2B = vacancy_vals.get("2B", 0)
            vac_2C = vacancy_vals.get("2C", 0)

            balloted_year = int(
                (app_2B > vac_2B) or (app_2C > vac_2C)
            )

            out.append({
                "school": current_school,
                "year": year,
                "applicants_2B": applied_vals.get("2B", 0),
                "applicants_2C": applied_vals.get("2C", 0),
                "vacancies_2B": vacancy_vals.get("2B", 0),
                "vacancies_2C": vacancy_vals.get("2C", 0),
                "taken_2B": taken_vals.get("2B", 0),
                "taken_2C": taken_vals.get("2C", 0),
                "balloted_year": balloted_year
            })

            vacancy_vals = None
            applied_vals = None
            taken_vals = None

    return out

# -------------------------
# Build full panel
# -------------------------
all_rows = []
for y in YEARS:
    rows = parse_year(y)
    print(f"{y}: {len(rows)} schools")
    all_rows.extend(rows)

df_2B2C_panel = pd.DataFrame(all_rows)

# remove duplicates if any
df_2B2C_panel = (
    df_2B2C_panel
    .groupby(["school", "year"], as_index=False)
    .first()
    .sort_values(["year", "school"])
    .reset_index(drop=True)
)

print(df_2B2C_panel.head(20))
print(df_2B2C_panel.shape)

# optional save
# df_2B2C_panel.to_csv("school_year_2B_2C_panel.csv", index=False)

2009: 168 schools
2010: 169 schools
2011: 170 schools
2012: 177 schools
2013: 180 schools
2014: 180 schools
2015: 183 schools
2016: 183 schools
2017: 184 schools
2018: 184 schools
2019: 185 schools
2020: 186 schools
2021: 181 schools
2022: 181 schools
2023: 181 schools
2024: 180 schools
2025: 179 schools
                     school  year  applicants_2B  applicants_2C  vacancies_2B  \
0                 Admiralty  2009             51            139            63   
1             Ahmad Ibrahim  2009              3             71            59   
2                   Ai Tong  2009             65             64            45   
3              Anchor Green  2009              0            110            95   
4                  Anderson  2009             25            116            57   
5                Ang Mo Kio  2009              2            107            65   
6    Anglo-Chinese (Junior)  2009             73             63            53   
7   Anglo-Chinese (Primary)  2009             

In [41]:
df_2B2C_panel

,school,year,applicants_2B,applicants_2C,vacancies_2B,vacancies_2C,taken_2B,taken_2C,balloted_year
0,Admiralty,2009,51,139,63,75,51,75,1
1,Ahmad Ibrahim,2009,3,71,59,116,3,70,0
2,Ai Tong,2009,65,64,45,48,45,48,1
3,Anchor Green,2009,0,110,95,189,0,110,0
4,Anderson,2009,25,116,57,88,25,88,1
...,...,...,...,...,...,...,...,...,...
3046,Yuhua,2025,1,26,58,174,1,26,0
3047,Yumin,2025,0,10,67,201,0,10,0
3048,Zhangde,2025,0,74,29,87,0,74,0
3049,Zhenghua,2025,0,50,20,62,0,50,0


In [42]:
df_2B2C_panel["RBF_st"] = (
    df_2B2C_panel.groupby("school")["balloted_year"]
      .transform(lambda s: s.rolling(window=5, min_periods=1).mean())
)

In [43]:
df_2B2C_panel['BI'] = np.log((df_2B2C_panel['applicants_2B']+df_2B2C_panel['applicants_2C'])/(df_2B2C_panel['vacancies_2B']+df_2B2C_panel['vacancies_2C']))

In [44]:
df_2B2C_panel["across_phase_ballot"] = (
    (df_2B2C_panel["applicants_2B"] > df_2B2C_panel["vacancies_2B"]) &
    (df_2B2C_panel["applicants_2C"] > df_2B2C_panel["vacancies_2C"])
).astype(int)

In [45]:
#We should start from 2013 omwards so that there is the full 5 years for each school year
df_2B2C_panel = df_2B2C_panel[df_2B2C_panel['year']>=2013]


In [46]:
#Normalise 
from scipy.stats import zscore
df_2B2C_panel["z_BI"] = 0.7 * zscore(
    df_2B2C_panel["BI"].astype(float),
    nan_policy="omit"
) + 0.3 * df_2B2C_panel['across_phase_ballot']

df_2B2C_panel['z_RBF'] = zscore(
    df_2B2C_panel["RBF_st"].astype(float),
    nan_policy="omit"
)

/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_75295/4206131706.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2B2C_panel["z_BI"] = 0.7 * zscore(
/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_75295/4206131706.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2B2C_panel['z_RBF'] = zscore(


In [47]:
df_2B2C_panel['SDI']= (df_2B2C_panel['z_RBF']+df_2B2C_panel['z_BI'])/2

/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_75295/3049317479.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2B2C_panel['SDI']= (df_2B2C_panel['z_RBF']+df_2B2C_panel['z_BI'])/2


In [48]:
df_2B2C_panel

,school,year,applicants_2B,applicants_2C,vacancies_2B,vacancies_2C,taken_2B,taken_2C,balloted_year,RBF_st,BI,across_phase_ballot,z_BI,z_RBF,SDI
684,Admiralty,2013,40,96,63,86,40,86,1,1.0,-0.091291,0,0.337804,1.110344,0.724074
685,Ahmad Ibrahim,2013,2,73,60,118,2,73,0,0.0,-0.864295,0,-0.275518,-1.096885,-0.686201
686,Ai Tong,2013,25,38,8,7,8,7,1,1.0,1.435085,1,1.848872,1.110344,1.479608
687,Alexandra,2013,0,232,105,210,0,210,1,1.0,-0.305835,0,0.167580,1.110344,0.638962
688,Anchor Green,2013,3,90,41,78,3,78,1,0.4,-0.246524,0,0.214639,-0.213993,0.000323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3046,Yuhua,2025,1,26,58,174,1,26,0,0.0,-2.150901,0,-1.296345,-1.096885,-1.196615
3047,Yumin,2025,0,10,67,201,0,10,0,0.0,-3.288402,0,-2.198870,-1.096885,-1.647877
3048,Zhangde,2025,0,74,29,87,0,74,0,0.2,-0.449525,0,0.053572,-0.655439,-0.300933
3049,Zhenghua,2025,0,50,20,62,0,50,0,0.0,-0.494696,0,0.017732,-1.096885,-0.539576


In [49]:
# 75th percentile SDI within each year
df_2B2C_panel["SDI_p75_year"] = df_2B2C_panel.groupby("year")["SDI"].transform(lambda x: x.quantile(0.75))

# label good schools: 1 if SDI >= year-specific 75th percentile, else 0
df_2B2C_panel["good_school"] = (df_2B2C_panel["SDI"] >= df_2B2C_panel["SDI_p75_year"]).astype(int)

/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_75295/3053648442.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2B2C_panel["SDI_p75_year"] = df_2B2C_panel.groupby("year")["SDI"].transform(lambda x: x.quantile(0.75))
/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_75295/3053648442.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2B2C_panel["good_school"] = (df_2B2C_panel["SDI"] >= df_2B2C_panel["SDI_p75_year"]).astype(int)


In [50]:
final_df = df_2B2C_panel[['school', 'year', 'SDI', 'good_school']]

In [51]:
final_df

,school,year,SDI,good_school
684,Admiralty,2013,0.724074,0
685,Ahmad Ibrahim,2013,-0.686201,0
686,Ai Tong,2013,1.479608,1
687,Alexandra,2013,0.638962,0
688,Anchor Green,2013,0.000323,0
...,...,...,...,...
3046,Yuhua,2025,-1.196615,0
3047,Yumin,2025,-1.647877,0
3048,Zhangde,2025,-0.300933,0
3049,Zhenghua,2025,-0.539576,0


In [52]:
final_df.to_csv("School_demand_index.csv", index=False)

In [53]:
top20 = (
    df_2B2C_panel.sort_values(["year", "SDI"], ascending=[True, False])
      .groupby("year")
      .head(20)
      .reset_index(drop=True)
)

In [54]:
top20

,school,year,applicants_2B,applicants_2C,vacancies_2B,vacancies_2C,taken_2B,taken_2C,balloted_year,RBF_st,BI,across_phase_ballot,z_BI,z_RBF,SDI,SDI_p75_year,good_school
0,Ai Tong,2013,25,38,8,7,8,7,1,1.0,1.435085,1,1.848872,1.110344,1.479608,0.785450,1
1,Henry Park,2013,9,25,5,5,5,5,1,1.0,1.223775,1,1.681214,1.110344,1.395779,0.785450,1
2,Nan Hua,2013,30,72,15,15,15,15,1,1.0,1.223775,1,1.681214,1.110344,1.395779,0.785450,1
3,Fairfield Methodist,2013,50,66,19,19,19,19,1,1.0,1.116004,1,1.595705,1.110344,1.353025,0.785450,1
4,CHIJ St. Nicholas Girls’,2013,33,37,13,12,13,12,1,1.0,1.029619,1,1.527165,1.110344,1.318755,0.785450,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,Rulang,2025,30,98,20,40,20,40,1,1.0,0.757686,1,1.311406,1.110344,1.210875,0.871393,1
256,Wellington,2025,21,104,20,40,20,40,1,1.0,0.733969,1,1.292589,1.110344,1.201466,0.871393,1
257,Methodist Girls’,2025,54,78,22,45,22,45,1,1.0,0.678109,1,1.248268,1.110344,1.179306,0.871393,1
258,CHIJ St. Nicholas Girls’,2025,48,71,20,41,20,41,1,1.0,0.668250,1,1.240445,1.110344,1.175394,0.871393,1


In [55]:
#top20.to_csv("top20_SDI.csv", index=False)

In [56]:
bottom_20  = (
    df_2B2C_panel.sort_values(["year", "SDI"], ascending=[True, True])
      .groupby("year")
      .head(20)
      .reset_index(drop=True)
)

In [64]:
bottom_20

,school,year,applicants_2B,applicants_2C,vacancies_2B,vacancies_2C,taken_2B,taken_2C,balloted_year,RBF_st,BI,across_phase_ballot,z_BI,z_RBF,SDI,SDI_p75_year,good_school
0,Damai,2013,0,15,62,125,0,15,0,0.0,-2.523058,0,-1.591625,-1.096885,-1.344255,0.785450,0
1,Juying,2013,1,23,95,188,1,23,0,0.0,-2.467393,0,-1.547459,-1.096885,-1.322172,0.785450,0
2,Blangah Rise,2013,0,21,70,140,0,21,0,0.0,-2.302585,0,-1.416696,-1.096885,-1.256790,0.785450,0
3,Teck Whye,2013,0,30,74,148,0,30,0,0.0,-2.001480,0,-1.177791,-1.096885,-1.137338,0.785450,0
4,Kranji,2013,1,34,73,144,1,34,0,0.0,-1.824549,0,-1.037409,-1.096885,-1.067147,0.785450,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,Ahmad Ibrahim,2025,1,31,62,185,1,31,0,0.0,-2.043652,0,-1.211252,-1.096885,-1.154068,0.871393,0
256,Farrer Park,2025,0,24,43,129,0,24,0,0.0,-1.969441,0,-1.152370,-1.096885,-1.124627,0.871393,0
257,Greendale,2025,0,24,43,128,0,24,0,0.0,-1.963610,0,-1.147744,-1.096885,-1.122314,0.871393,0
258,Changkat,2025,1,42,74,220,1,42,0,0.0,-1.922380,0,-1.115031,-1.096885,-1.105958,0.871393,0


In [58]:
#bottom_20.to_csv("old_bottom_20.csv", index=False)